# 06 — Scheduled Task Abuse for Persistence

| Field | Value |
|-------|-------|
| **MITRE ATT&CK ID** | T1053 — Scheduled Task/Job |
| **Tactic** | Execution / Persistence |
| **Difficulty** | Beginner |
| **Estimated Time** | 30 minutes |

## Threat Context: APT29 (Cozy Bear) — Scheduled Task Persistence

APT29, the Russian state-sponsored group behind the SolarWinds SUNBURST attack, extensively used Windows Scheduled Tasks and cron-style jobs for persistence. After gaining initial access, they created scheduled tasks that executed PowerShell payloads at system startup or on a recurring schedule, ensuring their implant survived reboots.

Scheduled tasks are a favored persistence mechanism because they are **legitimate OS features**, blend with normal system activity, and can be configured to run with elevated privileges. Tools like `schtasks.exe` (Windows) and `cron` (Linux) are present on every system and rarely blocked by application whitelisting.

## What You'll Understand After This

- How attackers use **scheduled tasks/cron jobs** for persistence and recurring execution
- The Python `sched` module as an analogy for **system-level task scheduling**
- How to **detect malicious scheduled tasks** through audit logs, task enumeration, and behavioral baselines

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Environment setup and imports

import sched
import time
import os
import json
import tempfile
from datetime import datetime, timedelta
from collections import OrderedDict

print("[+] Imports loaded successfully")
print("[i] This notebook uses Python's sched module to simulate scheduled task behavior.")
print("[i] No actual system scheduled tasks or cron jobs are created.")

### Step 1 — Understand Scheduled Task Structures

Both Windows (schtasks/Task Scheduler) and Linux (cron/systemd timers) allow recurring command execution. Attackers create tasks that run payloads at boot, login, or fixed intervals.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — scheduled task structure analysis

# Windows schtasks command that an attacker might use
windows_task_example = {
    "command": 'schtasks /create /tn "SystemHealthCheck" /tr "powershell -ep bypass -f C:\\Windows\\Temp\\update.ps1" /sc onlogon /ru SYSTEM',
    "task_name": "SystemHealthCheck",
    "trigger": "On user logon",
    "run_as": "SYSTEM",
    "payload": "C:\\Windows\\Temp\\update.ps1",
    "evasion": "Legitimate-sounding name, runs as SYSTEM"
}

# Linux cron entry that an attacker might use
linux_cron_example = {
    "entry": "*/15 * * * * /tmp/.cache/updater.sh > /dev/null 2>&1",
    "schedule": "Every 15 minutes",
    "payload": "/tmp/.cache/updater.sh",
    "evasion": "Hidden directory, output suppressed, generic name"
}

print("[+] Windows Scheduled Task Example:")
for key, val in windows_task_example.items():
    print(f"    {key}: {val}")

print("\n[+] Linux Cron Example:")
for key, val in linux_cron_example.items():
    print(f"    {key}: {val}")

### Step 2 — Simulate Scheduled Execution with Python

We use Python's `sched` module to simulate how a scheduled task runs a payload at fixed intervals, mimicking what a persistent implant does.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — simulate scheduled task execution

execution_log = []
scheduler = sched.scheduler(time.time, time.sleep)

def simulated_payload(task_name, run_number):
    """Simulates a scheduled task payload execution."""
    timestamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    entry = {
        "task": task_name,
        "run": run_number,
        "time": timestamp,
        "action": "beacon_simulated"  # In real attacks: C2 callback, data collection, etc.
    }
    execution_log.append(entry)
    print(f"  [{timestamp}] Task '{task_name}' — execution #{run_number}")

# Schedule 4 executions at 0.5-second intervals (simulating a recurring task)
print("[+] Scheduling simulated task executions...")
for i in range(4):
    scheduler.enter(0.5 * i, 1, simulated_payload, argument=("SystemHealthCheck", i + 1))

print("[+] Running scheduler (simulates recurring execution):\n")
scheduler.run()

print(f"\n[+] Total executions logged: {len(execution_log)}")

### Step 3 — Analyze Detection Patterns

Defenders look for newly created tasks, tasks with suspicious properties, and tasks that don't match the organization's baseline.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — scheduled task detection analysis

suspicious_indicators = [
    {"indicator": "Task runs from temp directory", "risk": "HIGH",
     "example": "/tmp/.cache/updater.sh"},
    {"indicator": "Task runs encoded PowerShell", "risk": "HIGH",
     "example": "powershell -enc <base64>"},
    {"indicator": "Task created outside business hours", "risk": "MEDIUM",
     "example": "Task created at 02:34 AM"},
    {"indicator": "Task runs as SYSTEM with network access", "risk": "HIGH",
     "example": "SYSTEM account making HTTP requests"},
    {"indicator": "Task name mimics legitimate service", "risk": "MEDIUM",
     "example": "WindowsUpdate, SystemHealthCheck"},
    {"indicator": "Task suppresses output", "risk": "LOW",
     "example": "> /dev/null 2>&1"},
]

print("[+] Suspicious Scheduled Task Indicators:")
print(f"{'Indicator':<45} {'Risk':<8} {'Example'}")
print("-" * 100)
for item in suspicious_indicators:
    print(f"{item['indicator']:<45} {item['risk']:<8} {item['example']}")

### Visualization — Scheduled Task Execution Timeline

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))

# Plot execution timeline
times = list(range(len(execution_log)))
labels = [f"Run #{e['run']}\n{e['time']}" for e in execution_log]

ax.scatter(times, [1] * len(times), s=200, color="#e74c3c", zorder=5, edgecolors="#c0392b")
ax.plot(times, [1] * len(times), color="#e74c3c", linewidth=2, alpha=0.5)

for i, label in enumerate(labels):
    ax.annotate(label, (times[i], 1), textcoords="offset points",
                xytext=(0, 20), ha="center", fontsize=8,
                arrowprops=dict(arrowstyle="->", color="gray"))

# Add scheduled task context
ax.set_xlim(-0.5, len(times) - 0.5)
ax.set_ylim(0.5, 1.8)
ax.set_xlabel("Execution Sequence")
ax.set_title("Simulated Scheduled Task Execution Timeline", fontsize=13, fontweight="bold")
ax.set_yticks([])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.tight_layout()
plt.show()
print("[+] Visualization complete")

## Defender's Perspective — Indicators of Compromise

| Indicator | Type | Detection Method |
|-----------|------|------------------|
| New scheduled task created (Event ID 4698) | System | Windows Security Event Log monitoring |
| Cron file modified outside change window | File | Linux auditd, file integrity monitoring |
| Task executing from `%TEMP%`, `/tmp`, or hidden directories | Behavioral | EDR process monitoring with path rules |
| Task created by non-admin user with SYSTEM privileges | Account | Privilege escalation detection |
| Recurring outbound connections at fixed intervals | Network | NetFlow analysis, beacon detection algorithms |

## Article Seed

**Title:** *Hiding in Plain Sight: How Attackers Abuse Scheduled Tasks for Persistence*

**Opening Paragraph:**
Every operating system ships with a built-in mechanism to run programs on a schedule — and attackers love it. Scheduled tasks and cron jobs provide reliable, stealthy persistence that blends seamlessly with legitimate system activity, making detection a challenge even for mature security operations centers.

**Section Headers:**
1. Scheduled Tasks 101: How Attackers Weaponize a System Feature
2. APT29 and the Art of Task-Based Persistence
3. Detection Engineering: Finding Malicious Tasks in the Noise
4. Hardening Scheduled Task Infrastructure

**Medium Tags:** `cybersecurity`, `persistence`, `scheduled-tasks`, `mitre-attack`, `threat-detection`

In [ ]:
# Self-check assertions

# 1. Verify scheduled executions occurred
assert len(execution_log) == 4, f"Expected 4 executions, got {len(execution_log)}"

# 2. Verify execution log structure
assert all("task" in e and "run" in e and "time" in e for e in execution_log), \
    "All execution log entries should have task, run, and time fields"

# 3. Verify detection indicators were defined
assert len(suspicious_indicators) >= 5, "Should define at least 5 suspicious indicators"

print("[+] All self-check assertions passed!")